In [68]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
import gensim
from gensim import corpora, models
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import re
from nltk.stem import WordNetLemmatizer
from gensim import corpora
from gensim.models.ldamodel import LdaModel
from collections import Counter
import pyLDAvis.gensim_models


In [69]:
df = pd.read_csv('gender_data_leapfunder.csv')

In [70]:
df = df[['position', 'Gender', 'bio']]

In [71]:
men_df = df[df['Gender'] == 'male']
women_df = df[df['Gender'] == 'female']

In [72]:
women_df = women_df.dropna()
men_df = men_df.dropna()

In [73]:
display(men_df.head(10))
display(women_df.head(10))

,position,Gender,bio
0,Chief Executive Officer,male,Jerome is one of the Co-Founders and CEO at Re...
1,CEO,male,"Robert Hoevers MSc, former GM at A1GP Netherla..."
2,Chief of Design,male,Chris Klok MSc MTD has a broad experience in d...
4,Founder & Developer,male,Joshua is a software architect and business de...
5,Developer & Devops,male,Humayun is the brain behind the logic of our a...
6,CEO/CTO,male,Serial entrepreneur who founded three tech sta...
7,Marketing & Sales,male,Founded two tech startups after helping to bui...
8,Founder,male,Serial founder
10,"Co-founder, CTO",male,"10+ yrs data engineering, systems architecture"
11,"AI/ML, Data Science",male,20 years quantitative modeling at large financ...


,position,Gender,bio
3,Designer & Branding,female,Danielle is our designer and branding queen.
9,"Co-founder, CEO",female,10+ yrs medical device & growth marketing
14,Hardware development & manufacturing partner,female,20+ years in hearables development and manufac...
18,Data Science,female,Elise has worked as a Data Scientist for vario...
20,Marketing,female,Eva is a digital marketeer that previously wor...
33,Designer & Branding,female,Danielle is our designer and branding queen.
39,"Co-founder, CEO",female,10+ yrs medical device & growth marketing
44,Hardware development & manufacturing partner,female,20+ years in hearables development and manufac...
48,Data Science,female,Elise has worked as a Data Scientist for vario...
50,Marketing,female,Eva is a digital marketeer that previously wor...


In [74]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    try:
        # remove special characters
        text = re.sub(r'[^A-Za-z\s]', '', text)
        # remove URLS
        text = re.sub(r'http\S+', '', text)
        # remove capitalisation
        text = text.lower()
        # tokenise
        text = text.split()
        preprocessed_text = []
        # remove stopwords + lemmatize
        for word in text:
            if word not in stop_words:
                preprocessed_text.append(word)
            word = lemmatizer.lemmatize(word)
    except:
        preprocessed_text = None
    return preprocessed_text


In [75]:
preprocessed_text_women = women_df['bio'].apply(preprocess_text)

dictionary_women = corpora.Dictionary(preprocessed_text_women)
corpus_women = [dictionary_women.doc2bow(text) for text in preprocessed_text_women]

In [76]:
num_topics = 10

lda_model_women = LdaModel(
    corpus=corpus_women,
    id2word=dictionary_women,
    num_topics=num_topics,
    passes=5
)

In [77]:
vis = pyLDAvis.gensim_models.prepare(topic_model=lda_model_women, dictionary=dictionary_women, corpus=corpus_women)
pyLDAvis.save_html(vis, 'bios_women.html')

In [78]:
preprocessed_text_men = men_df['bio'].apply(preprocess_text)

dictionary_men = corpora.Dictionary(preprocessed_text_men)
corpus_men = [dictionary_men.doc2bow(text) for text in preprocessed_text_men]

In [79]:
num_topics = 10

lda_model_men = LdaModel(
    corpus=corpus_men,
    id2word=dictionary_men,
    num_topics=num_topics,
    passes=5
)

In [80]:
vis = pyLDAvis.gensim_models.prepare(topic_model=lda_model_men, dictionary=dictionary_men, corpus=corpus_men)
pyLDAvis.save_html(vis, 'bios_men.html')